# Libraries

In [80]:
import pandas as pd
import numpy as np
import matplotlib as mpl 
import matplotlib.pyplot as plt
import seaborn as sns

In [81]:
path = "../Data/QVI_merged_tables.csv"
path = "/Users/ekaterinasorokopudova/Desktop/Quantium/Data/QVI_merged_tables.csv"

In [82]:
df = pd.read_csv(path)

df.head(5)

,DATE,STORE_NBR,LYLTY_CARD_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
0,2018-10-17,1,1000,1,5,natural chip compny seasalt175g,2,6.0,175,Natural Chip Co,YOUNG SINGLES/COUPLES,Premium
1,2019-05-14,1,1307,348,66,ccs nacho cheese 175g,3,6.3,175,CCs,MIDAGE SINGLES/COUPLES,Budget
2,2019-05-20,1,1343,383,61,smiths crinkle cut chips chicken 170g,2,2.9,170,Smiths,MIDAGE SINGLES/COUPLES,Budget
3,2018-08-17,2,2373,974,69,smiths chip thinly s/cream&onion 175g,5,15.0,175,Smiths,MIDAGE SINGLES/COUPLES,Budget
4,2018-08-18,2,2426,1038,108,kettle tortilla chpshny&jlpno chili 150g,3,13.8,150,Kettle,MIDAGE SINGLES/COUPLES,Budget


In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246739 entries, 0 to 246738
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATE              246739 non-null  object 
 1   STORE_NBR         246739 non-null  int64  
 2   LYLTY_CARD_NBR    246739 non-null  int64  
 3   TXN_ID            246739 non-null  int64  
 4   PROD_NBR          246739 non-null  int64  
 5   PROD_NAME         246739 non-null  object 
 6   PROD_QTY          246739 non-null  int64  
 7   TOT_SALES         246739 non-null  float64
 8   PACK_SIZE         246739 non-null  int64  
 9   BRAND             246739 non-null  object 
 10  LIFESTAGE         246739 non-null  object 
 11  PREMIUM_CUSTOMER  246739 non-null  object 
dtypes: float64(1), int64(6), object(5)
memory usage: 22.6+ MB


# Data analysis on customer segments 

- Who spends the most on chips (total sales), describing customers by lifestage 
and how premium their general purchasing behaviour is:

- How many customers are in each segment
- How many chips are bought per customer by segment
- What's the average chip price by customer segment

In [ ]:
avg_chips_customer = df["PROD_QTY"].sum() / df["LYLTY_CARD_NBR"].nunique()
avg_price_chip = df["TOT_SALES"].sum() / df["PROD_QTY"].sum()

In [90]:
segment_analysis = df.groupby("PREMIUM_CUSTOMER").agg(
    customers = ("LYLTY_CARD_NBR","nunique"),
    total_chips = ("PROD_QTY", "sum"),
    total_sales = ("TOT_SALES", "sum")
)

segment_analysis["chips_per_customer"] = (
    segment_analysis["total_chips"] /
    segment_analysis["customers"]
    ).round()

segment_analysis["consumption_index"] = (
    segment_analysis["chips_per_customer"] / avg_chips_customer
).round(2)

segment_analysis["avg_price_per_chips"] = (
    segment_analysis["total_sales"] /
    segment_analysis["total_chips"]
).round(2)

segment_analysis["price_index"] = (
    segment_analysis["avg_price_per_chips"] / avg_price_chip
).round(2)

segment_analysis = segment_analysis.sort_values(
    by= "customers", 
    ascending= False
)

segment_analysis

,customers,total_chips,total_sales,chips_per_customer,consumption_index,avg_price_per_chips,price_index
PREMIUM_CUSTOMER,,,,,,,
Mainstream,28734,180780,700865.40,6.0,0.91,3.88,1.01
Budget,24006,165774,631406.85,7.0,1.06,3.81,0.99
Premium,18547,123843,472899.45,7.0,1.06,3.82,1.00


#### Insight 1:

Chips appear to be a mass-market product with minimal price differentiation across customer segments. While Budget and Premium customers purchase slightly more chips per customer, price levels remain almost identical across segments.

In [91]:
lifestage_analysis = df.groupby("LIFESTAGE").agg(
    customers = ("LYLTY_CARD_NBR", "nunique"),
    total_chips = ("PROD_QTY", "sum"),
    total_sales = ("TOT_SALES", "sum")
)

lifestage_analysis["chips_per_customer"] = (
    lifestage_analysis["total_chips"] / 
    lifestage_analysis["customers"]
).round()

lifestage_analysis["consumption_index"] = (
    lifestage_analysis["chips_per_customer"] / avg_chips_customer
).round(2)

lifestage_analysis["avg_price_per_chips"] = (
    lifestage_analysis ["total_sales"] /
    lifestage_analysis ["total_chips"]
).round(2)

lifestage_analysis["price_index"] = (
    lifestage_analysis["avg_price_per_chips"] / avg_price_chip
).round(2)

lifestage_analysis = lifestage_analysis.sort_values(
    by= "customers", 
    ascending= False)

lifestage_analysis

,customers,total_chips,total_sales,chips_per_customer,consumption_index,avg_price_per_chips,price_index
LIFESTAGE,,,,,,,
RETIREES,14555,87875,342381.90,6.0,0.91,3.90,1.02
OLDER SINGLES/COUPLES,14389,97183,376013.65,7.0,1.06,3.87,1.01
YOUNG SINGLES/COUPLES,14044,62300,243756.60,4.0,0.61,3.91,1.02
OLDER FAMILIES,9630,87896,328519.90,9.0,1.36,3.74,0.97
YOUNG FAMILIES,9036,78577,294627.90,9.0,1.36,3.75,0.98
MIDAGE SINGLES/COUPLES,7141,44496,172523.80,6.0,0.91,3.88,1.01
NEW FAMILIES,2492,12070,47347.95,5.0,0.76,3.92,1.02


#### Insight 2: family households drive chip consumption

Families represent the most intensive consumers of chips. Both Young Families and Older Families purchase around 36% more chips per customer compared to the market average, although they tend to pay slightly below-average prices. This suggests that families buy chips in larger quantities and are more price-sensitive.

#### Insight 3: large segments do not necessarily drive consumption

Although Retirees and Young Singles/Couples represent some of the largest customer groups, their chip consumption per customer remains below the market average.

#### Insight 4: young singles are light snack consumers

Young Singles/Couples show the lowest consumption index (0.61), indicating significantly lower chip purchases compared to other life stages, despite paying slightly above-average prices.

In [93]:
analysis = df.groupby(["LIFESTAGE","PREMIUM_CUSTOMER"]).agg(
    customers = ("LYLTY_CARD_NBR", "nunique"),
    total_chips = ("PROD_QTY", "sum"),
    total_sales = ("TOT_SALES", "sum")
)

analysis["chips_per_customer"] = (
    analysis["total_chips"] / 
    analysis["customers"]
).round()

analysis["consumption_index"] = (
    analysis["chips_per_customer"] / avg_chips_customer
).round(2)

analysis["avg_price_per_chips"] = (
    analysis ["total_sales"] /
    analysis ["total_chips"]
).round(2)

analysis["price_index"] = (
    analysis["avg_price_per_chips"] / avg_price_chip
).round(2)

analysis

customers  total_chips  total_sales  \
LIFESTAGE              PREMIUM_CUSTOMER                                        
MIDAGE SINGLES/COUPLES Budget                 1474         8883     33345.70   
                       Mainstream             3298        21213     84734.25   
                       Premium                2369        14400     54443.85   
NEW FAMILIES           Budget                 1087         5241     20607.45   
                       Mainstream              830         4060     15979.70   
                       Premium                 575         2769     10760.80   
OLDER FAMILIES         Budget                 4611        41853    156863.75   
                       Mainstream             2788        25804     96413.55   
                       Premium                2231        20239     75242.60   
OLDER SINGLES/COUPLES  Budget                 4849        32883    127833.60   
                       Mainstream             4858        32607    124648.50   
                       Premium                4682        31693    123531.55   
RETIREES               Budget                 4385        26932    105916.30   
                       Mainstream             6358        37677    145168.95   
                       Premium                3812        23266     91296.65   
YOUNG FAMILIES         Budget                 3953        34482    129717.95   
                       Mainstream             2685        23194     86338.25   
                       Premium                2398        20901     78571.70   
YOUNG SINGLES/COUPLES  Budget                 3647        15500     57122.10   
                       Mainstream             7917        36225    147582.20   
                       Premium                2480        10575     39052.30   

                                         chips_per_customer  \
LIFESTAGE              PREMIUM_CUSTOMER                       
MIDAGE SINGLES/COUPLES Budget                           6.0   
                       Mainstream                       6.0   
                       Premium                          6.0   
NEW FAMILIES           Budget                           5.0   
                       Mainstream                       5.0   
                       Premium                          5.0   
OLDER FAMILIES         Budget                           9.0   
                       Mainstream                       9.0   
                       Premium                          9.0   
OLDER SINGLES/COUPLES  Budget                           7.0   
                       Mainstream                       7.0   
                       Premium                          7.0   
RETIREES               Budget                           6.0   
                       Mainstream                       6.0   
                       Premium                          6.0   
YOUNG FAMILIES         Budget                           9.0   
                       Mainstream                       9.0   
                       Premium                          9.0   
YOUNG SINGLES/COUPLES  Budget                           4.0   
                       Mainstream                       5.0   
                       Premium                          4.0   

                                         consumption_index  \
LIFESTAGE              PREMIUM_CUSTOMER                      
MIDAGE SINGLES/COUPLES Budget                         0.91   
                       Mainstream                     0.91   
                       Premium                        0.91   
NEW FAMILIES           Budget                         0.76   
                       Mainstream                     0.76   
                       Premium                        0.76   
OLDER FAMILIES         Budget                         1.36   
                       Mainstream                     1.36   
                       Premium                        1.36   
OLDER SINGLES/COUPLES  Budget                         1.06   
      

#### The main insight 1: 
Chip consumption patterns are primarily driven by life stage rather than by the customer's premium segment. Family households consistently purchase significantly more chips per customer, while singles and couples tend to purchase fewer packs regardless of their premium status.

#### The main insight 2: 
Premium status has limited impact on both purchase frequency and price paid in the chips category, suggesting that chips behave as a mass-market product across customer segments.

# The customer's total spend over the period and total spend

In [96]:
avg_per_txn = df["TOT_SALES"].sum() / df["TXN_ID"].count()

In [101]:
segment_analysis = df.groupby("PREMIUM_CUSTOMER").agg(
    customers = ("LYLTY_CARD_NBR","nunique"),
    total_txn = ("TXN_ID", "count"),
    total_sales = ("TOT_SALES", "sum")
)
segment_analysis["average bill"] = (
    segment_analysis["total_sales"] /
    segment_analysis["total_txn"]
    ).round(2)

segment_analysis["average_bill_index"] = (
    segment_analysis["average bill"] / avg_per_txn
).round(2)

segment_analysis = segment_analysis.sort_values(
    by= "customers", 
    ascending= False
)
segment_analysis

,customers,total_txn,total_sales,average bill,average_bill_index
PREMIUM_CUSTOMER,,,,,
Mainstream,28734,95043,700865.40,7.37,1.01
Budget,24006,86762,631406.85,7.28,1.00
Premium,18547,64934,472899.45,7.28,1.00
